In [ ]:
import torch
import os
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch.utils.data as Data
from torch.optim.lr_scheduler import CosineAnnealingLR,LinearLR, SequentialLR
import datetime
import Network_FDD_CBAM as FDD
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
import logging
import sys

# 创建日志文件，写入模式
log_filename = "training.log"
logging.basicConfig(
    level=logging.INFO,  # 输出所有 info 级别以上的信息
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filename, mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout)  # 同时输出到控制台
    ]
)

def log_print(*args, **kwargs):
    msg = " ".join(map(str, args))
    logging.info(msg)


In [ ]:
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')
if not os.path.exists('results'):
    os.makedirs('results')
DATA_DIR = "data"

In [ ]:
import subprocess
def temperatureCheck():
    try:
        # CPU
        cpu_temp = 95#psutil.sensors_temperatures().get('coretemp', [])[0].current if 'coretemp' in psutil.sensors_temperatures() else None
        # GPU
        gpu_query = subprocess.run(['nvidia-smi', '--query-gpu=temperature.gpu', '--format=csv,noheader,nounits'], capture_output=True, text=True)
        gpu_temp = int(gpu_query.stdout.strip())

        log_print(f"[TempCheck] GPU:{gpu_temp}°C")

        # 安全阈值
        # if cpu_temp and cpu_temp > 90:
        #     log_print("[TempCheck] CPU 过热，暂停 60 秒冷却")
        #     time.sleep(60)
        if gpu_temp and gpu_temp > 80:
            log_print("[TempCheck] GPU 过热，暂停 60 秒冷却")
            time.sleep(60)

    except Exception as e:
        log_print(f"[TempCheck] 获取温度信息失败: {e}")

In [ ]:
def set_seed(seed):
    """
    函数功能：为所有相关的库设置随机种子，以确保实验的可复现性
    """
    log_print(f"已设置种子为{seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def checkConvergence(epoch,
                     test_losses,
                     start_epoch=12,
                     window_size=10,
                     growth_threshold=0.1,
                     pass_score=15.0):
    """
    检查训练是否提前收敛或陷入停滞的函数。

    Args:
        epoch (int): 当前的训练轮次 (从0开始)。
        test_losses (list): 包含至今所有测试集性能（SE）的历史记录列表。
        start_epoch (int): 从第几轮开始进行检查。
        window_size (int): 需要连续多少轮的增长率都低于阈值才判断为收敛。
        growth_threshold (float): 增长率的阈值，低于此值被认为是“增长较小”。
        pass_score (float): 接受值，超过后不判断收敛

    Returns:
        bool: 如果判断为收敛，则返回 True，否则返回 False。
    """

    if epoch < start_epoch:
        return False
    if test_losses[-1] > pass_score:
        return False
    recent_performance = test_losses[-(window_size + 1):]
    for i in range(window_size):
        growth = recent_performance[-(i+1)] - recent_performance[-(i+2)]
        if growth >= growth_threshold:
            return False

    log_print(f"\n[Convergence Watcher] 在第 {epoch} 轮检测到性能停滞!")
    log_print(f"    - 已连续 {window_size} 轮的性能增长率低于阈值 ({growth_threshold})")
    log_print("    - 提前终止本次训练。")
    return True

In [ ]:
def train(Nc, N, Nt, B, Nr, L, SNR_dB, K, EPOCH, BATCH_SIZE):
    shoulian = np.zeros(EPOCH)
    snr = 10**(SNR_dB/10) / K
    parm_set = [Nc, Nt, Nr, snr, B, K]

    # H_train = torch.load(os.path.join(DATA_DIR, f'H_train_N{N}.pt'))
    # H_test = torch.load(os.path.join(DATA_DIR, f'H_test_N{N}.pt'))
    H_train_1 = torch.load('data/H_train_UPA' + str(N) + 'Lp_1.pt')[:, 0:K, :, :]
    H_train_2 = torch.load('data/H_train_UPA' + str(N) + 'Lp_2.pt')[:, 0:K, :, :]
    H_train = torch.cat([H_train_1, H_train_2], 0)
    H_test = torch.load('data/H_test_UPA' + str(N) + 'Lp.pt')[:, 0:K, :, :]

    net_US = FDD.DNN_US_RF_OFDM(parm_set).cuda()
    net_BS = FDD.DNN_BS_hyb_OFDM(parm_set).cuda()

    optimizer_US = torch.optim.Adam(net_US.parameters(), lr=0.001)
    scheduler_US = torch.optim.lr_scheduler.MultiStepLR(optimizer_US, milestones=[100, 150], gamma=0.6)
    optimizer_BS = torch.optim.Adam(net_BS.parameters(), lr=0.001)
    scheduler_BS = torch.optim.lr_scheduler.MultiStepLR(optimizer_BS, milestones=[100, 150], gamma=0.6)

    loss_func1 = FDD.MyLoss_OFDM().cuda()

    loader_train = Data.DataLoader(Data.TensorDataset(H_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    loader_test = Data.DataLoader(Data.TensorDataset(H_test), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    train_losses, test_losses = [], []
    best_test_se = 0
    plt.ion()  # 开启交互模式

    start = datetime.datetime.now()
    for epoch in range(EPOCH):
        temperatureCheck()

        train_SE, num_train = 0, 0
        test_SE, num_test = 0, 0

        for step, [b_x] in enumerate(loader_train):
            num_train += 1
            net_US.train()
            net_BS.train()
            b_x = b_x.cuda()
            num = b_x.shape[0]
            out1 = torch.zeros([num, B * K]).cuda()
            for i in range(K):
                out1[:, i * B:(i * B + B)] = net_US(b_x[:, i, :, :], parm_set)
            out2 = net_BS(out1, parm_set)
            loss = loss_func1(b_x, out2, parm_set)
            train_SE -= loss.item()

            optimizer_US.zero_grad()
            optimizer_BS.zero_grad()
            loss.backward()
            optimizer_US.step()
            optimizer_BS.step()

        train_SE /= num_train
        scheduler_US.step()
        scheduler_BS.step()

        net_US.eval()
        net_BS.eval()
        with torch.no_grad():
            for step, [b_x] in enumerate(loader_test):
                num_test += 1
                b_x = b_x.cuda()
                num = b_x.shape[0]
                out1 = torch.zeros([num, B * K]).cuda()
                for i in range(K):
                    out1[:, i * B:(i * B + B)] = net_US(b_x[:, i, :, :], parm_set)
                out2 = net_BS(out1, parm_set)
                loss = loss_func1(b_x, out2, parm_set)
                test_SE -= loss.item()

        test_SE /= num_test
        time_elapsed = datetime.datetime.now() - start
        log_print(f'Epoch: {epoch} | Time: {time_elapsed} | Train SE: {train_SE:.3f} | Test SE: {test_SE:.3f}')
        start = datetime.datetime.now()

        train_losses.append(train_SE)
        test_losses.append(test_SE)

        if epoch > 0 and epoch % 10 == 0:
            plt.clf()
            plt.plot(train_losses, label='Train SE')
            plt.plot(test_losses, label='Test SE')
            plt.xlabel('Epoch')
            plt.ylabel('Spectral Efficiency')
            plt.title(f'Training Progress (B={B}, N={N}) - Epoch {epoch}')
            plt.legend()
            plt.grid(True)
            plt.savefig(f'results/progress_snapshot_{B}B{N}Lp{L}L{K}K_epoch{epoch}.png')
            plt.pause(0.1)

        if test_SE > best_test_se:
            best_test_se = test_SE
            torch.save(net_US, f'saved_models/net_US_{B}B{N}Lp{L}L{K}K.pth')
            torch.save(net_BS, f'saved_models/net_BS_{B}B{N}Lp{L}L{K}K.pth')
            log_print(f'New best model saved with Test SE: {best_test_se:.4f}')

        shoulian[epoch] = test_SE

        if(checkConvergence(epoch,test_losses)):
            log_print(shoulian)
            return

    plt.ioff()
    plt.clf()
    plt.plot(train_losses, label='Train SE')
    plt.plot(test_losses, label='Test SE')
    plt.axhline(y=best_test_se, color='r', linestyle='--', label=f'Best Test SE: {best_test_se:.4f}')
    plt.xlabel('Epoch')
    plt.ylabel('Spectral Efficiency')
    plt.title(f'Final Training Progress (B={B}, N={N})')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'results/final_progress_{B}B{N}Lp{L}L{K}K.png')
    plt.show()
    log_print(f'The best SE is: {max(test_losses):.3f}')
    log_print(shoulian)

In [ ]:
# def train(Nc, N, Nt, B, Nr, L, SNR_dB, K, EPOCH, BATCH_SIZE):
#     shoulian = np.zeros(EPOCH)
#     snr = 10**(SNR_dB/10) / K
#     parm_set = [Nc, Nt, Nr, snr, B, K]
#
#     # H_train = torch.load(os.path.join(DATA_DIR, f'H_train_N{N}.pt'))
#     # H_test = torch.load(os.path.join(DATA_DIR, f'H_test_N{N}.pt'))
#     H_train_1 = torch.load('data2/H_train_UPA' + str(N) + 'Lp_1.pt')[:, 0:K, :, :]
#     H_train_2 = torch.load('data2/H_train_UPA' + str(N) + 'Lp_2.pt')[:, 0:K, :, :]
#     H_train = torch.cat([H_train_1, H_train_2], 0)
#     H_test = torch.load('data2/H_test_UPA' + str(N) + 'Lp.pt')[:, 0:K, :, :]
#
#     net_US = FDD.DNN_US_RF_OFDM(parm_set).cuda()
#     net_BS = FDD.DNN_BS_hyb_OFDM(parm_set).cuda()
#
#     optimizer_US = torch.optim.Adam(net_US.parameters(), lr=0.001)
#     scheduler_US = torch.optim.lr_scheduler.MultiStepLR(optimizer_US, milestones=[100, 150], gamma=0.6)
#     optimizer_BS = torch.optim.Adam(net_BS.parameters(), lr=0.001)
#     scheduler_BS = torch.optim.lr_scheduler.MultiStepLR(optimizer_BS, milestones=[100, 150], gamma=0.6)
#
#     loss_func1 = FDD.MyLoss_OFDM().cuda()
#
#     loader_train = Data.DataLoader(Data.TensorDataset(H_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
#     loader_test = Data.DataLoader(Data.TensorDataset(H_test), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
#
#     train_losses, test_losses = [], []
#     best_test_se = 0
#     plt.ion()  # 开启交互模式
#
#     start = datetime.datetime.now()
#     for epoch in range(EPOCH):
#         temperatureCheck()
#
#         net_US.train()
#         net_BS.train()
#         train_SE, num_train = 0, 0
#         test_SE, num_test = 0, 0
#
#         for step, [b_x] in enumerate(loader_train):
#             b_x = b_x.cuda()
#             batch_size = b_x.shape[0]
#             b_x_reshaped = b_x.view(batch_size * K, Nr, Nc, 2 * Nt)
#             out1_reshaped = net_US(b_x_reshaped, parm_set) # 输出 shape: [batch*K, B]
#             out1 = out1_reshaped.view(batch_size, K * B)
#             out2 = net_BS(out1, parm_set)
#             loss = loss_func1(b_x, out2, parm_set)
#             optimizer_US.zero_grad()
#             optimizer_BS.zero_grad()
#             loss.backward()
#             optimizer_US.step()
#             optimizer_BS.step()
#             train_SE -= loss.item()
#             num_train += 1
#
#         train_SE /= num_train
#         scheduler_US.step()
#         scheduler_BS.step()
#
#         net_US.eval()
#         net_BS.eval()
#         with torch.no_grad():
#             for step, [b_x] in enumerate(loader_test):
#                 num_test += 1
#                 b_x = b_x.cuda()
#                 num = b_x.shape[0]
#                 out1 = torch.zeros([num, B * K]).cuda()
#                 for i in range(K):
#                     out1[:, i * B:(i * B + B)] = net_US(b_x[:, i, :, :], parm_set)
#                 out2 = net_BS(out1, parm_set)
#                 loss = loss_func1(b_x, out2, parm_set)
#                 test_SE -= loss.item()
#
#         test_SE /= num_test
#         time_elapsed = datetime.datetime.now() - start
#         log_print(f'Epoch: {epoch} | Time: {time_elapsed} | Train SE: {train_SE:.3f} | Test SE: {test_SE:.3f}')
#         start = datetime.datetime.now()
#
#         train_losses.append(train_SE)
#         test_losses.append(test_SE)
#
#         if epoch > 0 and epoch % 10 == 0:
#             plt.clf()
#             plt.plot(train_losses, label='Train SE')
#             plt.plot(test_losses, label='Test SE')
#             plt.xlabel('Epoch')
#             plt.ylabel('Spectral Efficiency')
#             plt.title(f'Training Progress (B={B}, N={N}) - Epoch {epoch}')
#             plt.legend()
#             plt.grid(True)
#             plt.savefig(f'results2/progress_snapshot_{B}B{N}Lp{L}L{K}K_epoch{epoch}.png')
#             plt.pause(0.1)
#
#         if test_SE > best_test_se:
#             best_test_se = test_SE
#             torch.save(net_US, f'saved_models2/net_US_{B}B{N}Lp{L}L{K}K.pth')
#             torch.save(net_BS, f'saved_models2/net_BS_{B}B{N}Lp{L}L{K}K.pth')
#             log_print(f'New best model saved with Test SE: {best_test_se:.4f}')
#
#         shoulian[epoch] = test_SE
#
#         # if(checkConvergence(epoch,test_losses)):
#         #     log_print(shoulian)
#         #     return
#
#     plt.ioff()
#     plt.clf()
#     plt.plot(train_losses, label='Train SE')
#     plt.plot(test_losses, label='Test SE')
#     plt.axhline(y=best_test_se, color='r', linestyle='--', label=f'Best Test SE: {best_test_se:.4f}')
#     plt.xlabel('Epoch')
#     plt.ylabel('Spectral Efficiency')
#     plt.title(f'Final Training Progress (B={B}, N={N})')
#     plt.legend()
#     plt.grid(True)
#     plt.savefig(f'results2/final_progress_{B}B{N}Lp{L}L{K}K.png')
#     plt.show()
#     log_print(f'The best SE is: {max(test_losses):.3f}')
#     log_print(shoulian)

In [ ]:
Nc = 32 #number of subcarriers
N = 2   # Number of paths
Nt = 64 # Number of Antennas at the BS
Nr = 1  # Number of Antennas at the UE
B = 32

L = 8   # number of pilot OFDM symbols
SNR_dB = 25  # SNR
K = 2   # number of UEs
snr =  10**(SNR_dB/10)/K

BATCH_SIZE = 512
EPOCH = 180


In [ ]:
SEED = int(time.time())
set_seed(SEED)
train(Nc, N, Nt, B, Nr, L, SNR_dB, K, EPOCH, BATCH_SIZE)